# Tutorial: Supermix Colab Cloud Trainer v2

Audience:
- You, when you want the current Supermix Qwen training pipeline on Colab with the same model architecture, the same alignment stages, and cloud-safe resume.

Prerequisites:
- Colab runtime set to GPU.
- Google Drive available for persistent checkpoints and logs.
- Network access to GitHub and Hugging Face.

Learning goals:
- Reuse the same `source/qwen_supermix_pipeline.py` training path used in this repo.
- Keep checkpoints and caches alive across Colab restarts.
- Resume cleanly from the latest adapter checkpoint instead of losing progress.
- Track launches and checkpoints in a built-in SQLite registry on Drive.

Note:
- This keeps the same Qwen base model, LoRA/DoRA/RSLoRA stack, SFT stage, preference stage, distillation, and checkpoint layout. Colab will usually run the same recipe on CUDA rather than the local Windows setup.


## Outline

1. Verify the Colab runtime.
2. Mount Drive, define persistent paths, and create the run registry.
3. Pull a sparse copy of the repo into `/content`.
4. Install the missing training dependencies.
5. Build the current full training command with Colab-safe resume settings.
6. Launch training, stream logs, and record the run in SQLite.
7. Reattach, inspect checkpoints, and query the registry after reconnects.


In [ ]:
from __future__ import annotations

import platform
import shutil
import subprocess
import sys

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Colab is not attached to a GPU runtime. In Colab, open Runtime > Change runtime type > Hardware accelerator > GPU, then rerun from the top.')
subprocess.run(['nvidia-smi'], check=True)


## Step 1 - Mount Drive and set the persistent layout

The repo checkout stays in `/content` for speed. Checkpoints, logs, Hugging Face caches, and a lightweight SQLite run registry live on Drive so reconnects do not wipe them.


In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/supermix_colab_v2')
HF_HOME = DRIVE_ROOT / 'hf_cache'
LOG_DIR = DRIVE_ROOT / 'logs'
OUTPUT_DIR = DRIVE_ROOT / 'artifacts' / 'qwen_supermix_enhanced_v28_cloud_plus_colab'
RESUME_WARM_START_DIR = DRIVE_ROOT / 'artifacts' / 'qwen_supermix_enhanced_v26_full'
STATE_PATH = DRIVE_ROOT / 'last_training_launch_v2.json'
RUN_DB_PATH = DRIVE_ROOT / 'supermix_colab_runs.sqlite3'
WORKSPACE_DIR = Path('/content/Supermix_27')
REPO_WARM_START_DIR = WORKSPACE_DIR / 'artifacts' / 'qwen_supermix_enhanced_v26_full'
REPO_BUNDLED_WARM_START_DIR = WORKSPACE_DIR / 'dist' / 'SupermixQwenDesktopV26' / '_internal' / 'bundled_latest_artifact'

REPO_URL = 'https://github.com/kai9987kai/Supermix_27.git'
BRANCH = 'main'
HF_BASE_MODEL_REPO = 'Qwen/Qwen2.5-0.5B-Instruct'
HF_BASE_MODEL_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
BASE_MODEL = str(DRIVE_ROOT / 'base_models' / f'qwen2_5_0_5b_instruct_{HF_BASE_MODEL_REVISION}')
DEVICE = 'cuda'
SAVE_EVERY_STEPS = 40  # Match the existing Windows full-training launcher exactly.
TRAIN_PROFILE = 'cloud_plus'  # strict_parity or cloud_plus
ALLOW_BUNDLED_WARM_START_FALLBACK = False  # Set True only if you want to use the packaged v26_cognitive_activation adapter instead of exact v26_full.
EXTRA_ARGS: list[str] = []
PINNED_VERSIONS = {
    'torch': '2.4.1',
    'transformers': '5.2.0',
    'peft': '0.18.1',
    'accelerate': '1.12.0',
    'safetensors': '0.7.0',
    'matplotlib': '3.10.8',
    'pillow': '12.1.0',
    'nltk': '3.9.2',
    'tokenizers': '0.22.2',
    'huggingface_hub': '1.7.2',
}

for path in (DRIVE_ROOT, HF_HOME, LOG_DIR, OUTPUT_DIR, Path(BASE_MODEL).parent):
    path.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME / 'transformers')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME / 'hub')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHONHASHSEED'] = '48'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.pop('HF_HUB_OFFLINE', None)
os.environ.pop('TRANSFORMERS_OFFLINE', None)

if TRAIN_PROFILE == 'strict_parity':
    EXTRA_ARGS = [
        '--strict_determinism',
        '--disable_tf32',
        '--matmul_precision', 'highest',
    ]
elif TRAIN_PROFILE == 'cloud_plus':
    OUTPUT_DIR = DRIVE_ROOT / 'artifacts' / 'qwen_supermix_enhanced_v28_cloud_plus_colab'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    EXTRA_ARGS = [
        '--eval_split_mode', 'auto',
        '--sft_true_packing',
        '--sft_packing_max_samples_per_row', '2',
    ]
else:
    raise ValueError(f'Unsupported TRAIN_PROFILE: {TRAIN_PROFILE}')

{
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    'RESUME_WARM_START_DIR': str(RESUME_WARM_START_DIR),
    'WORKSPACE_DIR': str(WORKSPACE_DIR),
    'REPO_WARM_START_DIR': str(REPO_WARM_START_DIR),
    'REPO_BUNDLED_WARM_START_DIR': str(REPO_BUNDLED_WARM_START_DIR),
    'HF_BASE_MODEL_REPO': HF_BASE_MODEL_REPO,
    'HF_BASE_MODEL_REVISION': HF_BASE_MODEL_REVISION,
    'BASE_MODEL': BASE_MODEL,
    'TRAIN_PROFILE': TRAIN_PROFILE,
    'ALLOW_BUNDLED_WARM_START_FALLBACK': ALLOW_BUNDLED_WARM_START_FALLBACK,
    'RUN_DB_PATH': str(RUN_DB_PATH),
    'HF_HOME': str(HF_HOME),
}


## Step 2 - Sync the exact training inputs needed for the prior recipe

This repo is large. The notebook uses sparse checkout, but it still pulls the training code and datasets needed for the current full recipe.

- `source/` for the exact pipeline code.
- `datasets/` for the exact training corpora.
- `artifacts/qwen_supermix_enhanced_v26_full/` if that warm-start lineage exists in the repo snapshot.
- `dist/SupermixQwenDesktopV26/_internal/bundled_latest_artifact/` as an optional fallback warm-start source.

Important:
- If `v26_full` is not present in the repo snapshot, place that folder in Drive at `/content/drive/MyDrive/supermix_colab/artifacts/qwen_supermix_enhanced_v26_full` before running Step 3.


In [ ]:
import shutil
import subprocess


def run(cmd: list[str], cwd: Path | None = None) -> None:
    print('+', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True, cwd=str(cwd) if cwd else None)
    except subprocess.CalledProcessError as exc:
        joined = ' '.join(cmd)
        raise RuntimeError(f'Colab command failed with exit code {exc.returncode}: {joined}') from exc


if WORKSPACE_DIR.exists():
    print(f'Removing existing workspace at {WORKSPACE_DIR} to avoid shallow-checkout state issues...')
    shutil.rmtree(WORKSPACE_DIR)

run([
    'git',
    'clone',
    '--filter=blob:none',
    '--sparse',
    '--depth',
    '1',
    '--branch',
    BRANCH,
    REPO_URL,
    str(WORKSPACE_DIR),
])

run(['git', 'sparse-checkout', 'set', 'source', 'datasets', 'artifacts/qwen_supermix_enhanced_v26_full', 'dist/SupermixQwenDesktopV26/_internal/bundled_latest_artifact'], cwd=WORKSPACE_DIR)
if not REPO_WARM_START_DIR.exists():
    print('Repo snapshot does not contain artifacts/qwen_supermix_enhanced_v26_full. Step 3 will require that warm-start folder in Drive instead.')
if REPO_BUNDLED_WARM_START_DIR.exists():
    print(f'Bundled warm-start fallback found at {REPO_BUNDLED_WARM_START_DIR}')

if shutil.which('git-lfs') is None:
    run(['apt-get', 'update'])
    run(['apt-get', 'install', '-y', 'git-lfs'])
run(['git', 'lfs', 'install'], cwd=WORKSPACE_DIR)
run([
    'git',
    'lfs',
    'pull',
    '--include=datasets/**,artifacts/qwen_supermix_enhanced_v26_full/**,dist/SupermixQwenDesktopV26/_internal/bundled_latest_artifact/**',
], cwd=WORKSPACE_DIR)

run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'source/requirements_train_build.txt'], cwd=WORKSPACE_DIR)
run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    *(f'{name}=={version}' for name, version in PINNED_VERSIONS.items()),
    'sentencepiece',
    'tqdm',
], cwd=WORKSPACE_DIR)

from importlib.metadata import version
from huggingface_hub import snapshot_download
import torch

resolved_versions = {name: version(name) for name in PINNED_VERSIONS}
print('Pinned versions:', resolved_versions)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available after dependency installation.')
print('GPU:', torch.cuda.get_device_name(0))

snapshot_download(
    repo_id=HF_BASE_MODEL_REPO,
    revision=HF_BASE_MODEL_REVISION,
    local_dir=BASE_MODEL,
    local_dir_use_symlinks=False,
)
print('Workspace ready:', WORKSPACE_DIR)
print('Warm-start artifact path:', REPO_WARM_START_DIR)
print('Pinned base model path:', BASE_MODEL)


## Step 3 - Build the current full recipe with resume support

This mirrors the existing `source/run_train_qwen_supermix_v26_full.ps1` recipe.

- `--device cuda` is forced because this notebook is specifically for Colab GPU.
- The warm-start lineage is enforced. If the `v26_full` adapter is missing, the notebook stops instead of silently starting a different training path.
- `TRAIN_PROFILE='strict_parity'` is the default and keeps the exact prior recipe.
- `TRAIN_PROFILE='research_plus'` is optional and should only be used for a separate experiment output, not for exact parity.


In [ ]:
import json
import os
import shutil
from datetime import datetime, timezone

LOGICAL_CPU = max(1, os.cpu_count() or 1)
INTEROP_CPU = max(1, min(4, LOGICAL_CPU // 2))

DATA_FILES = [
    'datasets/conversation_data.quality_anchor_v2.jsonl',
    'datasets/conversation_data.coding_knowledge_2026_02_19.jsonl',
    'datasets/conversation_data.world_events_2026_02_19.jsonl',
    'datasets/conversation_data.supermix_plus_v27_500k.jsonl',
    'datasets/conversation_data.mega_reasoning_creative_v25_75582.jsonl',
    'datasets/conversation_data.mega_creative_250k_v2.jsonl',
]


def get_latest_adapter_checkpoint(run_dir: Path) -> Path | None:
    if not run_dir.exists():
        return None
    direct_adapter = run_dir / 'adapter'
    if (direct_adapter / 'adapter_config.json').exists() and (direct_adapter / 'adapter_model.safetensors').exists():
        return direct_adapter.resolve()
    if (run_dir / 'adapter_config.json').exists() and (run_dir / 'adapter_model.safetensors').exists():
        return run_dir.resolve()
    latest_file = run_dir / 'latest_adapter_checkpoint.txt'
    if latest_file.exists():
        checkpoint_dir = latest_file.read_text(encoding='utf-8').strip()
        if checkpoint_dir:
            raw = Path(checkpoint_dir)
            candidates = [raw]
            if not raw.is_absolute():
                candidates.append(WORKSPACE_DIR / raw)
                candidates.append(run_dir / raw)
            for candidate in candidates:
                if candidate.exists():
                    return candidate.resolve()
    checkpoints_dir = run_dir / 'checkpoints'
    if not checkpoints_dir.exists():
        return None
    metas = sorted(checkpoints_dir.rglob('checkpoint_meta.json'), key=lambda path: path.stat().st_mtime, reverse=True)
    for meta in metas:
        adapter_dir = meta.parent / 'adapter'
        if adapter_dir.exists():
            return adapter_dir.resolve()
    return None


def seed_drive_warm_start() -> Path | None:
    drive_checkpoint = get_latest_adapter_checkpoint(RESUME_WARM_START_DIR)
    if drive_checkpoint is not None:
        return drive_checkpoint
    repo_checkpoint = get_latest_adapter_checkpoint(REPO_WARM_START_DIR)
    if repo_checkpoint is None:
        return None
    print('Seeding Drive warm-start artifact from repo checkout...')
    shutil.copytree(REPO_WARM_START_DIR, RESUME_WARM_START_DIR, dirs_exist_ok=True)
    return get_latest_adapter_checkpoint(RESUME_WARM_START_DIR)


def get_bundled_fallback_checkpoint() -> Path | None:
    return get_latest_adapter_checkpoint(REPO_BUNDLED_WARM_START_DIR)


def build_train_command() -> list[str]:
    cmd = [
        sys.executable,
        '-u',
        'source/qwen_supermix_pipeline.py',
        '--data',
        *DATA_FILES,
        '--base_model',
        BASE_MODEL,
        '--output_dir',
        str(OUTPUT_DIR),
        '--max_records',
        '600000',
        '--max_source_fraction',
        '0.52',
        '--max_synthetic_fraction',
        '0.06',
        '--max_prompt_signature_count',
        '4',
        '--data_log_every_records',
        '2000',
        '--prompt_signature_cap_exempt_sources',
        'conversation_data.quality_anchor_v2.jsonl,conversation_data.mega_reasoning_creative_v25_75582.jsonl',
        '--eval_size',
        '500',
        '--eval_min_quality_score',
        '1.05',
        '--eval_drop_synthetic_prompts',
        '--max_length',
        '448',
        '--batch_size',
        '1',
        '--grad_accum_steps',
        '16',
        '--epochs',
        '6',
        '--max_steps',
        '6200',
        '--lr',
        '1.0e-5',
        '--sft_lr_schedule',
        'cosine_restarts',
        '--sft_lr_restart_period',
        '620',
        '--sft_warmup_steps',
        '30',
        '--sft_min_lr_ratio',
        '0.22',
        '--sft_max_grad_norm',
        '0.9',
        '--sft_focal_gamma',
        '1.35',
        '--sft_eval_every_steps',
        '240',
        '--sft_early_stop_patience',
        '5',
        '--sft_curriculum_quality_ramp',
        '0.22',
        '--sft_grad_noise_eta',
        '0.01',
        '--train_log_every_steps',
        '1',
        '--save_every_steps',
        str(SAVE_EVERY_STEPS),
        '--weight_decay',
        '0.02',
        '--lora_r',
        '32',
        '--lora_alpha',
        '64',
        '--lora_dropout',
        '0.03',
        '--use_rslora',
        '--use_dora',
        '--lora_init',
        'pissa_niter_4',
        '--lora_plus_ratio',
        '16',
        '--neftune_noise_alpha',
        '5.0',
        '--sft_weight_mode',
        'quality',
        '--sft_min_weight',
        '0.62',
        '--sft_max_weight',
        '1.88',
        '--sft_synthetic_prompt_weight',
        '0.62',
        '--sft_teacher_source_weight',
        '0.92',
        '--sft_quality_anchor_boost',
        '1.14',
        '--sft_coding_boost',
        '1.24',
        '--sft_events_boost',
        '1.08',
        '--sft_reasoning_boost',
        '1.28',
        '--sft_prompt_skill_boost',
        '1.17',
        '--sft_conversation_boost',
        '1.24',
        '--sft_creativity_boost',
        '1.16',
        '--sft_knowledge_density_boost',
        '1.22',
        '--sft_rdrop_alpha',
        '0.05',
        '--sft_length_bucketed_batches',
        '--sft_length_bucket_window_mult',
        '24',
        '--sft_followup_paraphrase_aug',
        '1',
        '--sft_followup_paraphrase_weight',
        '0.68',
        '--sft_min_quality_score',
        '0.98',
        '--sft_quality_filter_exempt_sources',
        'conversation_data.quality_anchor_v2.jsonl,conversation_data.world_events_2026_02_19.jsonl',
        '--sft_drop_synthetic_prompts',
        '--sft_auto_balance_sources',
        '--sft_source_balance_strength',
        '0.66',
        '--sft_source_balance_max_scale',
        '1.95',
        '--preference_objective',
        'ipo',
        '--preference_steps',
        '1500',
        '--preference_rescore_every',
        '25',
        '--preference_pairs',
        '34000',
        '--preference_candidate_count',
        '8',
        '--preference_reject_similarity_min',
        '0.16',
        '--preference_beta',
        '1.9',
        '--preference_beta_end',
        '3.6',
        '--preference_margin',
        '0.00',
        '--preference_margin_end',
        '0.00',
        '--preference_label_smoothing',
        '0.03',
        '--preference_sft_weight',
        '0.32',
        '--preference_length_weight',
        '0.08',
        '--preference_hardness_gamma',
        '1.15',
        '--preference_robust_alpha',
        '0.30',
        '--preference_robust_eta',
        '0.08',
        '--preference_robust_clip',
        '2.5',
        '--preference_wpo_alpha',
        '0.35',
        '--preference_wpo_clip',
        '2.5',
        '--preference_reference_anchor_weight',
        '0.04',
        '--preference_reference_anchor_batch_size',
        '2',
        '--preference_short_reject_boost',
        '0.75',
        '--preference_long_reject_boost',
        '0.25',
        '--preference_min_chosen_quality',
        '0.92',
        '--preference_min_chosen_words',
        '8',
        '--preference_min_quality_gap',
        '0.05',
        '--preference_allow_template_prompts',
        '--preference_max_pairs_per_user',
        '2',
        '--preference_max_pairs_per_source',
        '360',
        '--preference_mining_mode',
        'auto',
        '--preference_mining_progress_every',
        '30',
        '--preference_mining_max_seconds',
        '4500',
        '--preference_mining_max_attempt_factor',
        '20',
        '--preference_coding_focus_boost',
        '1.30',
        '--preference_reasoning_focus_boost',
        '1.32',
        '--preference_counterfactual_rejects_per_prompt',
        '4',
        '--preference_selection_strategy',
        'innovation_mix',
        '--preference_selection_keep_ratio',
        '0.62',
        '--preference_selection_min_keep',
        '1800',
        '--preference_selection_max_keep',
        '2400',
        '--preference_selection_hardness_target',
        '0.46',
        '--preference_selection_hardness_bandwidth',
        '0.22',
        '--preference_length_bucketed_batches',
        '--preference_length_bucket_window_mult',
        '24',
        '--preference_lr',
        '1.4e-5',
        '--preference_lr_schedule',
        'cosine',
        '--preference_warmup_steps',
        '18',
        '--preference_min_lr_ratio',
        '0.30',
        '--preference_max_grad_norm',
        '0.9',
        '--preference_max_new_tokens',
        '112',
        '--preference_prompt_max_tokens',
        '352',
        '--supermix_distill_ratio',
        '0.14',
        '--supermix_distill_max',
        '8000',
        '--supermix_distill_best_of',
        '3',
        '--supermix_distill_log_every',
        '40',
        '--supermix_distill_max_seconds',
        '12000',
        '--supermix_distill_min_quality',
        '0.93',
        '--supermix_distill_min_gain',
        '0.18',
        '--supermix_distill_density_bias',
        '0.20',
        '--seed',
        '48',
        '--device',
        DEVICE,
        '--device_preference',
        'cuda,npu,xpu,mps,cpu,dml',
        '--model_dtype',
        'auto',
        '--gradient_checkpointing',
        '--torch_num_threads',
        str(LOGICAL_CPU),
        '--torch_interop_threads',
        str(INTEROP_CPU),
        '--skip_benchmark',
    ]
    latest_output_checkpoint = get_latest_adapter_checkpoint(OUTPUT_DIR)
    if latest_output_checkpoint is not None:
        print(f'Resuming from latest checkpoint in {OUTPUT_DIR}')
        cmd.append('--resume_from_latest_checkpoint')
    else:
        warm_start_checkpoint = seed_drive_warm_start()
        if warm_start_checkpoint is not None:
            print(f'Warm-starting from checkpoint in {RESUME_WARM_START_DIR}')
            cmd.extend(['--init_adapter_dir', str(warm_start_checkpoint), '--init_adapter_match_lora'])
        else:
            bundled_fallback = get_bundled_fallback_checkpoint() if ALLOW_BUNDLED_WARM_START_FALLBACK else None
            if bundled_fallback is not None:
                print(f'Using bundled warm-start fallback from {REPO_BUNDLED_WARM_START_DIR}')
                cmd.extend(['--init_adapter_dir', str(bundled_fallback), '--init_adapter_match_lora'])
            else:
                raise RuntimeError(
                    f'Exact v26_full warm-start adapter was not found. Put that folder at {RESUME_WARM_START_DIR} in Drive, then rerun Step 3. If you want a non-identical fallback, set ALLOW_BUNDLED_WARM_START_FALLBACK = True to use the packaged v26_cognitive_activation adapter from {REPO_BUNDLED_WARM_START_DIR}.'
                )
    if EXTRA_ARGS:
        print('Applying extra args:', EXTRA_ARGS)
        cmd.extend(EXTRA_ARGS)
    return cmd


TRAIN_CMD = build_train_command()
OUT_LOG = LOG_DIR / f'train_{OUTPUT_DIR.name}.out.log'

launch_state = {
    'updated_at_utc': datetime.now(timezone.utc).isoformat(),
    'repo_url': REPO_URL,
    'branch': BRANCH,
    'workspace_dir': str(WORKSPACE_DIR),
    'output_dir': str(OUTPUT_DIR),
    'resume_warm_start_dir': str(RESUME_WARM_START_DIR),
    'base_model': BASE_MODEL,
    'device': DEVICE,
    'out_log': str(OUT_LOG),
    'command': TRAIN_CMD,
}
STATE_PATH.write_text(json.dumps(launch_state, indent=2), encoding='utf-8')
print(json.dumps(launch_state, indent=2))


## Step 4 - Launch training

This cell streams training logs into the notebook and also appends them to Drive. If Colab disconnects, rerun the previous cell and then rerun this one. The pipeline will resume from the latest checkpoint under `OUTPUT_DIR`.


In [ ]:
import shlex

print('Command preview:')
print(' '.join(shlex.quote(part) for part in TRAIN_CMD))
print('Streaming to:', OUT_LOG)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

with OUT_LOG.open('a', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        TRAIN_CMD,
        cwd=str(WORKSPACE_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f'Training exited with code {return_code}')

print('Training finished successfully.')


## Step 5 - Reattach and inspect progress

Use this after reconnects to confirm that checkpoints are being written and to read the latest log tail without reopening the full log file.


In [ ]:
from collections import deque

latest_checkpoint = get_latest_adapter_checkpoint(OUTPUT_DIR)
print('Latest checkpoint:', latest_checkpoint)

checkpoints_dir = OUTPUT_DIR / 'checkpoints'
if checkpoints_dir.exists():
    recent_checkpoints = sorted(checkpoints_dir.glob('*'), key=lambda path: path.stat().st_mtime, reverse=True)[:5]
    print('Recent checkpoint dirs:')
    for checkpoint_dir in recent_checkpoints:
        print(' -', checkpoint_dir)
else:
    print('No checkpoint directory yet.')

if OUT_LOG.exists():
    with OUT_LOG.open('r', encoding='utf-8', errors='replace') as f:
        tail = deque(f, maxlen=80)
    print(''.join(tail))
else:
    print('No log file yet.')
